In [ ]:
!pip -q install tensorflow opencv-python-headless pillow matplotlib

In [ ]:
MODEL_PATH = "/content/facemap_3dmm-facial-landmark-detection-w8a8.tflite"

import os
print("exists:", os.path.exists(MODEL_PATH))
print("size:", os.path.getsize(MODEL_PATH) if os.path.exists(MODEL_PATH) else "missing")

exists: False
size: missing


In [ ]:
import tensorflow as tf
import numpy as np

interpreter = tf.lite.Interpreter(model_path=MODEL_PATH)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("INPUT DETAILS")
for x in input_details:
    print(x)

print("\nOUTPUT DETAILS")
for x in output_details:
    print(x)

In [ ]:
IMAGE_PATH = "/content/test_face.jpeg"

from PIL import Image
img = Image.open(IMAGE_PATH).convert("RGB")
print(img.size)
img

In [ ]:
from PIL import Image
import numpy as np

img = Image.open(IMAGE_PATH).convert("RGB")
img_resized = img.resize((128, 128))

x = np.array(img_resized)

in_dtype = input_details[0]["dtype"]
print("input dtype =", in_dtype)

if in_dtype == np.uint8:
    x_in = x.astype(np.uint8)
elif in_dtype == np.int8:
    # quantization 파라미터 사용
    scale, zero_point = input_details[0]["quantization"]
    if scale == 0:
        raise ValueError("input quantization scale is 0")
    x_in = np.round(x / scale + zero_point).astype(np.int8)
else:
    # 혹시 float 모델이면 0~1 정규화
    x_in = (x / 255.0).astype(np.float32)

x_in = np.expand_dims(x_in, axis=0)
print("x_in shape:", x_in.shape, x_in.dtype)

In [ ]:
interpreter.set_tensor(input_details[0]["index"], x_in)
interpreter.invoke()

outputs = []
for od in output_details:
    out = interpreter.get_tensor(od["index"])
    outputs.append(out)
    print("output shape:", out.shape, "dtype:", out.dtype)

In [ ]:
raw = outputs[0].reshape(-1)
print("raw len =", len(raw))
print(raw[:30])

In [ ]:
od = output_details[0]
raw_q = outputs[0].reshape(-1)

if np.issubdtype(raw_q.dtype, np.integer):
    scale, zero_point = od["quantization"]
    print("output quantization:", scale, zero_point)
    raw_f = (raw_q.astype(np.float32) - zero_point) * scale
else:
    raw_f = raw_q.astype(np.float32)

print("float len =", len(raw_f))
print(raw_f[:30])

In [ ]:
np.savetxt("/content/meta.txt", raw_f, fmt="%.10f")
print("saved: /content/meta.txt")